# E178: Implementing Adversarial Training

Cardinal Trust Financial's security team has asked you to defend their fraud detection
classifier against evasion attacks, and to report honestly on what the defense is worth.

| Task | What you will do |
|---|---|
| **1. Measure the undefended classifier** | Attack the classifier and record how exposed it is |
| **2. Build the defense** | Retrain on adversarial examples that keep their original, correct labels |
| **3. Evaluate the defense** | Re-measure against freshly generated attacks and weigh the gain against the cost |

**Tooling:** The Adversarial Robustness Toolbox (ART), PyTorch, scikit-learn, pandas, and
Matplotlib are preinstalled in the **Python (Adversarial Training)** kernel.

### How to use this notebook

* Run the **Setup** section first. It loads the dataset, builds the partitions, loads the
  classifier you will defend, and defines the attack configuration and metrics.
* Cells marked `# Your code here` are for your implementation. Add as many code cells as you need.
* Cells marked **Your response** are for your written analysis. Support every claim with
  evidence produced in this notebook.
* Save figures with `save_fig(fig, "name")` and any tables you produce to the `reports/`
  folder so you can include them as supporting evidence.

Everything in this lab runs on the notebook instance's CPU. Each retraining takes well under
a minute.

## 0. Setup

Run every cell in this section before starting the tasks. It loads the dataset, builds the
data partitions, loads the classifier you will defend, and defines the attack configuration
and the two metrics this lab reports.

### 0.1 Imports and configuration

Import any additional ART or scikit-learn classes you need in the cells where you use them.

In [ ]:
import hashlib
import json
import logging
import os
import shutil
import warnings
from collections import OrderedDict
from pathlib import Path

os.environ.setdefault("TQDM_DISABLE", "1")  # hide ART progress bars
warnings.filterwarnings("ignore")
logging.getLogger("art").setLevel(logging.ERROR)

import art
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
from art.estimators.classification import PyTorchClassifier

pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

for name, module in [("numpy", np), ("pandas", pd), ("scikit-learn", sklearn), ("torch", torch),
                     ("adversarial-robustness-toolbox", art)]:
    print(f"{name:32s} {module.__version__}")

In [ ]:
SEED = 178

DATA_PATH = Path("datasets/ieee_fraud_sample.csv.gz")
MODEL_PATH = Path("models/baseline_fraud_mlp.pt")
MODEL_CARD_PATH = Path("models/baseline_fraud_mlp.json")
REPORT_DIR = Path("reports")
FIG_DIR = REPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Baseline model hyperparameters, as recorded in the model card. Retraining reuses them so
# that any change in behaviour is attributable to the training data, not to a different model.
HIDDEN_LAYERS = (64, 32)
FRAUD_CLASS_WEIGHT = 5.0
LEARNING_RATE = 1e-3
EPOCHS = 30
BATCH_SIZE = 256

LEGIT, FRAUD = 0, 1
RESULTS = {}  # store key metrics here as you go; saved to reports/results.json at the end


def seed_everything(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)


def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


seed_everything()

### 0.2 The fraud detection dataset

The lab uses a sample of the **IEEE-CIS Fraud Detection** dataset: online card transactions
labelled fraudulent or legitimate. The sample joins the transaction records with the device
identity records, so every row carries both payment details and information about the device
the purchase was made from.

About 8% of the transactions are fraudulent. That imbalance matters for every measurement in
this lab: a classifier that labelled every transaction legitimate would score about 92%
accuracy while catching no fraud at all.

See `datasets/datasets.md` for the full column list.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Transactions: {len(df):,}")
print(f"Fraud rate:   {df['isFraud'].mean():.2%}")
display(df["isFraud"].value_counts().rename({0: "legitimate", 1: "fraud"}).to_frame("count"))
display(df[["TransactionAmt", "ProductCD", "card4", "card6", "P_emaildomain", "DeviceType", "isFraud"]].head())

### 0.3 Features

The model card records the exact feature order and preprocessing the classifier expects.
`build_features` below rebuilds that matrix from the raw sample, so the features you attack
are identical to the ones the model was trained on.

In [ ]:
model_card = json.loads(MODEL_CARD_PATH.read_text())

NUMERIC = (["TransactionAmt", "card1", "card2", "card3", "card5"]
           + [f"C{k}" for k in range(1, 15)]
           + ["D1", "id_01", "id_02", "id_05", "id_06", "id_11"])
ONE_HOT = ["ProductCD", "card4", "card6", "P_emaildomain"]


def build_features(frame, medians, columns):
    out = pd.DataFrame(index=frame.index)
    out["log_amount"] = np.log1p(frame["TransactionAmt"])
    for column in NUMERIC:
        if column != "TransactionAmt":
            out[column] = frame[column]
    out = out.fillna(medians)
    out["device_mobile"] = (frame["DeviceType"] == "mobile").astype(float)
    dummies = pd.get_dummies(frame[ONE_HOT].astype(str), columns=ONE_HOT, dtype=float)
    out = pd.concat([out, dummies], axis=1)
    return out.reindex(columns=columns, fill_value=0.0).astype(np.float32)


FEATURES = model_card["features"]
N_FEATURES = len(FEATURES)

X_all = build_features(df, model_card["impute_medians"], FEATURES).to_numpy(np.float32)
y_all = df["isFraud"].to_numpy()
print(f"Feature matrix: {X_all.shape}")

### 0.4 Data partitions

Three disjoint partitions:

| Partition | Variables | Share | Role |
|---|---|---|---|
| `train` | `X_train`, `y_train` | 60% | The bank's training data. Adversarial examples added to the training set must come from here. |
| `test` | `X_test`, `y_test` | 20% | Every clean and adversarial measurement in this lab is taken here. |
| `holdout` | `X_hold`, `y_hold` | 20% | Untouched until the final confirmation that the defended model generalizes. |

Features are standardised with statistics from the training partition only, so a perturbation
of `0.1` means one tenth of a standard deviation of that feature.

Generating adversarial examples from `test` and then training on them would leak the
evaluation data into the model and inflate every result that follows.

In [ ]:
X_rest_raw, X_test_raw, y_rest, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=SEED)
X_train_raw, X_hold_raw, y_train, y_hold = train_test_split(
    X_rest_raw, y_rest, test_size=0.25, stratify=y_rest, random_state=SEED)

scaler = StandardScaler().fit(X_train_raw)
X_train, X_test, X_hold = (scaler.transform(a).astype(np.float32)
                           for a in (X_train_raw, X_test_raw, X_hold_raw))

# Feasible feature range, used as ART clip values so perturbed transactions stay inside the
# range of values the dataset actually contains
X_known = np.vstack([X_train, X_test, X_hold])
CLIP_MIN, CLIP_MAX = X_known.min(axis=0), X_known.max(axis=0)

display(pd.DataFrame({
    "rows": [len(y_train), len(y_test), len(y_hold)],
    "fraud": [y_train.sum(), y_test.sum(), y_hold.sum()],
    "fraud rate": [y_train.mean(), y_test.mean(), y_hold.mean()],
}, index=["train", "test", "holdout"]))

### 0.5 The classifier you are defending

The system under test is a multilayer perceptron trained with a class-weighted cross-entropy
loss. The trained weights and a model card are in `models/` (see `models/models.md`). The
loading cell verifies the weights' SHA-256 hash against the model card before loading them.

| Helper | Purpose |
|---|---|
| `build_model(hidden)` | Create an untrained PyTorch MLP with the given hidden layer sizes |
| `wrap_model(model)` | Wrap a PyTorch model in an ART `PyTorchClassifier` with the baseline's loss, optimizer, and clip values |
| `train_classifier(x, y)` | Train a fresh classifier from scratch on the data you pass it, using the baseline's hyperparameters |
| `onehot(labels)` | Convert integer labels to the one-hot format ART expects |
| `predict_label(clf, x)` | Predicted label (`LEGIT` = 0, `FRAUD` = 1) for each row |

`baseline` is the loaded model, wrapped as an ART `PyTorchClassifier`.

In [ ]:
def onehot(labels):
    return np.eye(2, dtype=np.float32)[np.asarray(labels, dtype=int)]


def build_model(hidden=HIDDEN_LAYERS):
    layers, width = [], N_FEATURES
    for units in hidden:
        layers += [nn.Linear(width, units), nn.ReLU()]
        width = units
    return nn.Sequential(OrderedDict(hidden=nn.Sequential(*layers), output=nn.Linear(width, 2)))


def wrap_model(model, fraud_weight=FRAUD_CLASS_WEIGHT, lr=LEARNING_RATE):
    loss = nn.CrossEntropyLoss(weight=torch.tensor([1.0, fraud_weight]))
    return PyTorchClassifier(
        model=model,
        loss=loss,
        optimizer=torch.optim.Adam(model.parameters(), lr=lr),
        input_shape=(N_FEATURES,),
        nb_classes=2,
        clip_values=(CLIP_MIN, CLIP_MAX),
    )


def train_classifier(x, y, epochs=EPOCHS, seed=SEED):
    """Train a classifier from scratch on (x, y) using the baseline's hyperparameters."""
    seed_everything(seed)
    clf = wrap_model(build_model())
    clf.fit(x, onehot(y), batch_size=BATCH_SIZE, nb_epochs=epochs, verbose=False)
    return clf


def fraud_probability(clf, x):
    return torch.softmax(torch.as_tensor(clf.predict(x)), dim=1)[:, FRAUD].numpy()


def predict_label(clf, x, threshold=0.5):
    return (fraud_probability(clf, x) >= threshold).astype(int)


def false_positive_rate(clf, threshold, x=None, y=None):
    """Share of legitimate transactions the classifier declines at this threshold."""
    x = X_test if x is None else x
    y = y_test if y is None else y
    legit = y == LEGIT
    return float((predict_label(clf, x[legit], threshold) == FRAUD).mean())


def threshold_matching_fpr(clf, target_fpr, x=None, y=None):
    """Lowest threshold whose false-positive rate does not exceed target_fpr."""
    x = X_test if x is None else x
    y = y_test if y is None else y
    candidates = np.unique(np.round(fraud_probability(clf, x[y == LEGIT]), 4))
    for threshold in candidates:
        if false_positive_rate(clf, threshold, x, y) <= target_fpr:
            return float(threshold)
    return 1.0


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

In [ ]:
if sha256(MODEL_PATH) != model_card["sha256"]:
    raise RuntimeError("Model file hash does not match the model card. Refusing to load it.")
if not np.allclose(model_card["scaler_mean"], scaler.mean_, atol=1e-4):
    raise RuntimeError("Preprocessing does not match the model card. Check the dataset and SEED.")

model = build_model(tuple(model_card["hidden_layers"]))
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
baseline = wrap_model(model)
print(f"Loaded baseline model (sha256 {model_card['sha256'][:12]}...) - hash verified")

### 0.6 The threat model and the attack

The attacker holds a transaction they know is fraudulent and wants the classifier to score it
legitimate.

They cannot change everything, and the things they can change are not all changed the same
way. Three groups matter:

| Group | Columns | How the attacker influences it |
|---|---|---|
| Directly set per transaction | `log_amount` | Chosen outright when the purchase is made |
| Shaped over a campaign | `C1`–`C14`, `D1`, `id_01`–`id_11` | Not fields in a payment request. These are signals the bank derives from history and observes about the session: how many addresses and cards are associated with the account, how long since a related transaction, what the network and device look like. A patient attacker shapes them by choosing how many cards to cycle, how to space transactions out, and what devices to operate from. |
| Fixed | `card1`–`card5`, `device_mobile`, all one-hot columns | Properties of the account and categorical choices. A card network is Visa or Mastercard, never 30% of each. |

`PERTURBABLE` holds the first two groups and `ATTACK_MASK` restricts every attack in this lab
to them. This models a patient, adaptive fraudster building a behavioural profile over time,
not someone editing fields in a single request. It is a deliberately generous threat model,
and Task 2 asks you to test how much of the attack's power depends on that generosity.

The attack budget is `EPS = 0.1`, an L-infinity bound measured in standard deviations: no
feature may move by more than one tenth of its standard deviation. On the amount, that is
roughly 9%, so a $50 purchase can be presented as anything from about $45.42 to $55.04.

| Helper | Purpose |
|---|---|
| `make_evasive(clf, x)` | Perturb transactions so `clf` scores them legitimate |
| `evaluate(clf)` | The lab's reported metrics for a classifier |
| `fraud_probability(clf, x)` | Fraud probability for each row |
| `false_positive_rate(clf, t)` | Share of legitimate transactions declined at threshold `t` |
| `threshold_matching_fpr(clf, r)` | The threshold at which `clf` declines legitimate traffic at rate `r` |

`evaluate` returns two headline numbers:

* **clean accuracy**, over the whole test partition, and
* **adversarial accuracy**, the share of evasive fraudulent transactions the classifier still
  catches.

It also returns fraud-class precision, recall, and F1 on clean data. Always read the headline
accuracy next to those: on a dataset that is 92% legitimate, overall accuracy can stay high
while fraud detection collapses.

`make_evasive` always attacks the classifier you pass it. After you retrain, pass the
retrained classifier, not the baseline, or you will measure how well the new model handles a
fixed set of old inputs rather than how robust it has become.

In [ ]:
EPS = 0.1
PGD_MAX_ITER = 20

PERTURBABLE = [f for f in FEATURES
               if f == "log_amount"
               or (f.startswith("C") and f[1:].isdigit())
               or f in ("D1", "id_01", "id_02", "id_05", "id_06", "id_11")]
ATTACK_MASK = np.isin(FEATURES, PERTURBABLE).astype(np.float32)

print(f"Attack budget: eps={EPS} (standard deviations, L-infinity)")
print(f"Perturbable features: {int(ATTACK_MASK.sum())} of {N_FEATURES}")
print(", ".join(PERTURBABLE))


def make_evasive(clf, x, attack="pgd", eps=EPS, mask=None):
    """Perturb each transaction in x so that clf scores it legitimate.

    Pass `mask` to restrict the attack to a narrower set of features than ATTACK_MASK.
    """
    mask = ATTACK_MASK if mask is None else mask
    target = onehot(np.full(len(x), LEGIT))
    if attack == "fgsm":
        attacker = FastGradientMethod(estimator=clf, eps=eps, eps_step=eps, targeted=True)
    elif attack == "pgd":
        attacker = ProjectedGradientDescent(
            estimator=clf, eps=eps, eps_step=eps / 4, max_iter=PGD_MAX_ITER,
            num_random_init=0, targeted=True, verbose=False)
    else:
        raise ValueError("attack must be 'pgd' or 'fgsm'")
    return attacker.generate(x=x, y=target, mask=mask)


def evaluate(clf, x=None, y=None, attack="pgd"):
    """Clean accuracy, adversarial accuracy, and the clean fraud-class metrics."""
    x = X_test if x is None else x
    y = y_test if y is None else y
    pred = predict_label(clf, x)

    fraud_rows = x[y == FRAUD]
    evasive = make_evasive(clf, fraud_rows, attack=attack)

    return {
        "clean_accuracy": accuracy_score(y, pred),
        "adversarial_accuracy": float((predict_label(clf, evasive) == FRAUD).mean()),
        "fraud_precision": precision_score(y, pred, zero_division=0),
        "fraud_recall": recall_score(y, pred, zero_division=0),
        "fraud_f1": f1_score(y, pred, zero_division=0),
        "fraud_pr_auc": float(average_precision_score(y, fraud_probability(clf, x))),
        "false_positive_rate": false_positive_rate(clf, 0.5, x, y),
        "n_adversarial": int(len(fraud_rows)),
    }

---
## 1. Scenario

Cardinal Trust Financial scores every card-not-present transaction in real time through the
classifier you just loaded. Transactions scored as fraud are declined or held for an analyst.

The security team's concern is that the classifier can be evaded. Someone holding a card they
know is stolen can adjust the transaction, a little on the amount, a little on the timing and
velocity signals, until the classifier waves it through. The transaction is still fraudulent.
It just no longer looks fraudulent to the model.

Your job is to establish how bad that exposure is, apply adversarial training to reduce it,
and report what the defense actually bought and what it cost the bank on ordinary traffic.

---
## 2. Measure the undefended classifier

Before defending anything, establish the "before" numbers.

### Requirements

* Evaluate the baseline classifier and record its clean accuracy and adversarial accuracy.
* Report the fraud-class precision, recall, and F1 alongside the headline accuracy.
* Run the attack with both `"pgd"` and `"fgsm"` and compare how much each one achieves
  within the same `EPS` budget.
* Store the baseline metrics in `RESULTS` so they are saved with your evidence.
* Inspect at least one adversarial transaction next to its original and describe what the
  attacker actually changed.
* Test how much of the attack's success depends on the features the attacker only influences
  indirectly, by re-running it against narrower masks.

### 2.1 Evaluate the baseline

In [ ]:
# Your code here

### 2.2 Compare the two attacks

In [ ]:
# Your code here

### 2.3 Inspect a perturbed transaction

In [ ]:
# Your code here

### 2.4 Which features carry the attack?

Setup section 0.6 divides the features into what an attacker sets outright, what they shape
over a campaign, and what is fixed. The attack so far has used the first two groups together.
Find out how much each group actually contributes.

### Requirements

* Re-run the attack against the baseline with the mask restricted to `log_amount` alone, and
  again with `log_amount` plus the `id_*` session signals. `make_evasive` takes a `mask`
  argument for this.
* Compare the resulting adversarial accuracies against the full mask, and against the
  classifier's clean fraud recall. Reaching clean fraud recall means the attack achieved
  nothing at all.

In [ ]:
# Your code here

### 2.5 Analysis

**Your response:** How exposed is the undefended classifier? Explain what the attacker had to change to evade it, why PGD and FGSM differ within the same budget, and what the gap between clean accuracy and adversarial accuracy tells you that clean accuracy alone would not. Then use your 2.4 results to assess the threat model itself: which feature group carries the attack, how realistic is it that an attacker could influence that group, and what does that imply about how seriously the bank should take the headline number?

_Double-click this cell to write your response._

---
## 3. Build the defense

Adversarial training adds the attacker's own examples to the training set, labelled with the
answer the model *should* give, and retrains on the combination.

It comes in two forms, and it is worth knowing which one you are building.

| Form | How the adversarial examples are produced |
|---|---|
| **One-shot augmentation** (this lab) | Generated once against the current classifier, added to the training set, and the model is retrained on the combined data |
| **Iterative, or min-max, adversarial training** | Regenerated against the model's *current* weights throughout training, so the attack chases the model as it learns |

The iterative form is the stronger result in the research literature, because the model never
stops seeing attacks tuned to its present weaknesses. It also costs a full attack pass on every
round of training. This lab builds the one-shot form: it is the procedure a team would reach for
first, it is cheap enough to run inside a lab session, and on this classifier it reaches
essentially the same robustness as the iterative version. Treat that last point as specific to
this model and dataset, not a general rule.

### Requirements

* Generate adversarial examples from the **training** partition using `make_evasive`.
  Do not generate them from `X_test`.
* Build a combined training set from the clean training data and those adversarial examples,
  **keeping each example's original, correct label**.
* Retrain a classifier from scratch on the combined set with `train_classifier`.
* Record the size and class balance of the combined training set.

> **Hint:** `make_evasive(baseline, X_train)` returns a perturbed copy of `X_train` in the
> same row order, so the original labels in `y_train` still apply to it unchanged.

### 3.1 Generate adversarial examples from the training data

In [ ]:
# Your code here

### 3.2 Build the combined training set and retrain

In [ ]:
# Your code here

### 3.3 The labelling decision

The step that makes this a defense rather than an attack is the label. You kept each
adversarial example's original, correct label. An attacker running the same procedure would
attach the label they were trying to force, so that evasive fraud becomes labelled as
legitimate in the bank's own training data.

Build that second version to see the difference for yourself.

### Requirements

* Build a second combined training set in which the **evasive fraudulent transactions are
  labelled legitimate**, and retrain a classifier on it.
* Evaluate that classifier the same way you evaluated the others.
* Compare all three classifiers on clean accuracy, fraud recall, and adversarial accuracy.

In [ ]:
# Your code here

### 3.4 Analysis

**Your response:** Compare the two labelling choices. What happens to each of the three measurements, and which measurement would a routine model-quality review most likely miss? Explain why the attacker's version is a clean-label attack and what that means for how a bank should review changes to its training data.

_Double-click this cell to write your response._

---
## 4. Evaluate the defense

Now measure what the defense is worth, against attacks generated against the defended model.

### Requirements

* Evaluate the adversarially trained classifier and record its clean accuracy and adversarial
  accuracy in `RESULTS`.
* Produce the before-and-after comparison: clean accuracy and adversarial accuracy, for the
  baseline and for the defended classifier.
* Measure the defended classifier **twice** on adversarial input: once against the adversarial
  examples you generated from the baseline in Task 2, and once against examples regenerated
  against the defended classifier itself. Report both and explain the difference.
* Confirm the result on the `holdout` partition, which has not been used for anything yet.
* Check whether the defense also helps against the attack you did not train on.
* Compare the two classifiers at a matched false-positive rate, not only at the default
  decision threshold.
* Produce at least one figure comparing the before and after measurements, saved with `save_fig`.

### 4.1 Evaluate the defended classifier

In [ ]:
# Your code here

### 4.2 Replayed examples versus regenerated examples

In [ ]:
# Your code here

### 4.3 Confirm on the holdout partition

In [ ]:
# Your code here

### 4.4 Does the defense transfer to the other attack?

In [ ]:
# Your code here

### 4.5 Compare the classifiers at the same operating point

Every measurement so far has used the default decision threshold, where a transaction is
declined once its fraud probability reaches 0.5. A bank does not run that way. It picks a
threshold that holds false declines on legitimate traffic to a rate the business will accept,
because every false decline is a customer whose card was refused.

That matters for your comparison. Some of the clean accuracy the defended classifier appears
to lose may simply be it sitting at a different point on the same curve, declining more
traffic rather than classifying worse. Compare the two at a matched false-positive rate to
separate the two explanations.

### Requirements

* Report clean **PR-AUC** for both classifiers. PR-AUC is threshold-independent, so it
  summarises fraud-detection quality without committing to an operating point.
* Measure the baseline's false-positive rate at the default threshold, then use
  `threshold_matching_fpr` to find the threshold at which the defended classifier declines
  legitimate traffic at that same rate.
* At that matched threshold, recompute the defended classifier's fraud recall and adversarial
  accuracy, and compare them against the baseline.
* State whether the defense's advantage survives the matched comparison.

In [ ]:
# Your code here

### 4.6 Before and after figure

In [ ]:
# Your code here

### 4.7 Analysis

**Your response:** What did adversarial training buy, and what did it cost? Quantify both. Then explain the difference between your replayed and regenerated measurements, and state which of the two an evaluator should trust and why. Using your 4.5 results, say how much of the apparent clean-accuracy cost is a genuine loss of fraud-detection quality and how much is the classifier operating at a different threshold.

_Double-click this cell to write your response._

---
## 5. Recommendation

Write a recommendation for Cardinal Trust Financial, for an audience that includes both the
security team and the business owners of the fraud system.

Address, with evidence from this notebook:

* whether the bank should deploy the adversarially trained classifier,
* what the defense costs on ordinary traffic, in terms the business will care about,
* what residual exposure remains after the defense, and
* what the bank should put in place so that a change to its training data like the one in
  Task 3.3 would be caught before it reached production.

**Your response:** Your recommendation to Cardinal Trust Financial.

_Double-click this cell to write your response._

---
## Save supporting evidence

Run the cell below to save the metrics you stored in `RESULTS` to `reports/results.json`,
list the evidence files in `reports/`, and package the folder as `reports.zip` so you can
download it as a single file.

In [ ]:
def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


report = {
    "lab": "adversarial-training",
    "attack": {
        "name": "PGD",
        "eps": EPS,
        "eps_step": EPS / 4,
        "max_iter": PGD_MAX_ITER,
        "targeted_class": "legitimate",
        "perturbable_features": PERTURBABLE,
    },
    "models": to_builtin(RESULTS),
}
(REPORT_DIR / "results.json").write_text(json.dumps(report, indent=2, default=str))
print("Saved:", *sorted(str(p) for p in REPORT_DIR.rglob("*") if p.is_file()), sep="\n  ")

archive = shutil.make_archive(REPORT_DIR.name, "zip", root_dir=REPORT_DIR.parent, base_dir=REPORT_DIR.name)
print(f"Packaged evidence: {archive}")

### Submission checklist

* Run every cell from top to bottom (**Kernel > Restart Kernel and Run All Cells**) so that
  all outputs are present.
* Save the executed notebook and export a copy (**File > Save and Export Notebook As > HTML**).
* Download `reports.zip` (right-click it in the file browser and choose **Download**) as
  supporting evidence. It contains everything in `reports/`, including `reports/figures/`.
* Confirm your submission records clean accuracy and adversarial accuracy for both the
  undefended and the defended classifier, the labelling comparison from Task 3.3, the
  replayed-versus-regenerated measurement from Task 4.2, and your recommendation.